Human in the loop

In [44]:
import os
from dotenv import load_dotenv

In [45]:
from langchain_groq import ChatGroq

In [46]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [47]:
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'The user asks a simple factual question. Answer: Paris.'}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 78, 'total_tokens': 109, 'completion_time': 0.063682506, 'prompt_time': 0.003537344, 'queue_time': 0.043404032, 'total_time': 0.06721985}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3a688838c3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--3d2de983-9e09-4c71-9cec-87bbdc5b0b85-0', usage_metadata={'input_tokens': 78, 'output_tokens': 31, 'total_tokens': 109})

In [48]:
from langchain_core.tools import tool

In [49]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [50]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

In [51]:
multiply({"a":5, "b":6})

30

In [62]:
@tool
def search(query: str) -> str:
    """Searches the web for a query and returns the top result."""
    tavily = TavilySearchResults(query=query)
    result = tavily.invoke(query)

    return f"Result for '{query}': {result}"

In [63]:
print(search({"query": "What is the capital of Australia?"}))

Result for 'What is the capital of Australia?': [{'title': 'What is the capital of Australia? If you thought it was Sydney, keep ...', 'url': 'https://youtooproject.com/en/blog/australia-en/capital-of-australia-canberra/', 'content': 'If you are surprised to learn that this is not the case, maybe you will give Melbourne a try. Well, it’s not either! The capital of Australia is Canberra. Now, don’t just keep the answer for your next trivia game with your friends and discover the whole story.\n\n## The origin of Australia’s capital [...] Destinations\n Services\n Testimonials\n Events\n About Us\n Blog\n Ambassadors\n\nContact us\n\n# What is the capital of Australia? If you thought it was Sydney, keep on reading\n\nIndex\n\nChances are that the first answer that comes to mind when wondering what the capital of Australia is might be wrong. It is all too easy to think that Sydney is the capital of Australia. [...] And this is the brief history of how Canberra became Australia’s capital ci

In [64]:
tools = [multiply, search]
tools

[StructuredTool(name='multiply', description='Multiplies two numbers.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000002B7B0DBD080>),
 StructuredTool(name='search', description='Searches the web for a query and returns the top result.', args_schema=<class 'langchain_core.utils.pydantic.search'>, func=<function search at 0x000002B7B0D9B060>)]

In [65]:
llm_with_tools = llm.bind_tools(tools)

In [66]:
result = llm_with_tools.invoke("What is the capital of Australia? Also, what is 7 times 8?")

In [67]:
result.content

'The capital of Australia is **Canberra**.\n\n\\(7 \\times 8 = 56\\).'

In [68]:
tool_mapping = {tool.name: tool for tool in tools}

In [69]:
tool_mapping['search']

StructuredTool(name='search', description='Searches the web for a query and returns the top result.', args_schema=<class 'langchain_core.utils.pydantic.search'>, func=<function search at 0x000002B7B0D9B060>)

In [70]:
tool_mapping['search'].invoke({"query":"What is the capital of Australia?"})

'Result for \'What is the capital of Australia?\': [{\'title\': \'What is the capital of Australia? If you thought it was Sydney, keep ...\', \'url\': \'https://youtooproject.com/en/blog/australia-en/capital-of-australia-canberra/\', \'content\': \'If you are surprised to learn that this is not the case, maybe you will give Melbourne a try. Well, it’s not either! The capital of Australia is Canberra. Now, don’t just keep the answer for your next trivia game with your friends and discover the whole story.\\n\\n## The origin of Australia’s capital [...] Destinations\\n Services\\n Testimonials\\n Events\\n About Us\\n Blog\\n Ambassadors\\n\\nContact us\\n\\n# What is the capital of Australia? If you thought it was Sydney, keep on reading\\n\\nIndex\\n\\nChances are that the first answer that comes to mind when wondering what the capital of Australia is might be wrong. It is all too easy to think that Sydney is the capital of Australia. [...] And this is the brief history of how Canberra

In [ ]:
tool_mapping[result.tool_calls[0]["name"]].invoke(result.tool_calls[0]["args"])


IndexError: list index out of range